In [2]:
import pandas as pd

val_df = pd.read_csv("/Users/rohan/Documents/Final year/cwe_project/scripts/template/juliet_train_template_split.csv")

print("Validation samples:", len(val_df))
print("Unique CWEs:", val_df["cwe"].nunique())

Validation samples: 144109
Unique CWEs: 118


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = "codet5_juliet_multiclass"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

device = "mps" if torch.backends.mps.is_available() else "cpu"
model.to(device)
model.eval()

/opt/homebrew/Caskroom/miniconda/base/envs/cwe/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1222 00:52:56.880000 5523 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


T5ForSequenceClassification(
  (transformer): T5Model(
    (shared): Embedding(32100, 512)
    (encoder): T5Stack(
      (embed_tokens): Embedding(32100, 512)
      (block): ModuleList(
        (0): T5Block(
          (layer): ModuleList(
            (0): T5LayerSelfAttention(
              (SelfAttention): T5Attention(
                (q): Linear(in_features=512, out_features=512, bias=False)
                (k): Linear(in_features=512, out_features=512, bias=False)
                (v): Linear(in_features=512, out_features=512, bias=False)
                (o): Linear(in_features=512, out_features=512, bias=False)
                (relative_attention_bias): Embedding(32, 8)
              )
              (layer_norm): T5LayerNorm()
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (1): T5LayerFF(
              (DenseReluDense): T5DenseActDense(
                (wi): Linear(in_features=512, out_features=2048, bias=False)
                (wo): Linear(in_featu

In [4]:
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report

class JulietDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(int(self.labels[idx]))
        }

dataset = JulietDataset(
    val_df["code_clean"],
    val_df["cwe_index"],
    tokenizer
)

loader = DataLoader(dataset, batch_size=8)

all_preds, all_labels = [], []

with torch.no_grad():
    for batch in loader:
        outputs = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device)
        )
        preds = outputs.logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch["labels"].numpy())

print(classification_report(all_labels, all_preds, digits=4))

              precision    recall  f1-score   support

           0     0.9955    0.9910    0.9932      1336
           1     0.9997    0.9993    0.9995     12158
           2     0.9876    0.9994    0.9934     12700
           3     0.8421    0.9697    0.9014       462
           4     0.9927    0.9982    0.9954      4356
           5     1.0000    0.9994    0.9997      3206
           6     0.9943    0.9991    0.9967      4356
           7     0.8962    0.9954    0.9432      7782
           8     0.0000    0.0000    0.0000       152
           9     1.0000    1.0000    1.0000       156
          10     1.0000    0.7692    0.8696        78
          11     0.9998    0.9983    0.9990      9178
          12     1.0000    0.9984    0.9992      7342
          13     0.9420    0.9887    0.9648      2824
          14     0.9681    0.9873    0.9776      2824
          15     0.0000    0.0000    0.0000        42
          16     0.9975    0.9916    0.9945      2380
          17     1.0000    

/opt/homebrew/Caskroom/miniconda/base/envs/cwe/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/Caskroom/miniconda/base/envs/cwe/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/Caskroom/miniconda/base/envs/cwe/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _war

In [ ]:
from sklearn.metrics import classification_report
import pandas as pd

report = classification_report(
    all_labels,
    all_preds,
    digits=4,
    output_dict=True
)
report_df = pd.DataFrame(report).transpose()
report_df.reset_index(inplace=True)
report_df.rename(columns={"index": "cwe_index"}, inplace=True)

# Keep only actual classes (remove accuracy / avg rows)
report_df = report_df[
    report_df["cwe_index"].str.isnumeric()
]
report_df["cwe_index"] = report_df["cwe_index"].astype(int)
# Load mapping
mapping_df = pd.read_csv("/Users/rohan/Documents/Final year/cwe_project/juliet_cwe_dataset.csv")
cwe_map = mapping_df[["cwe_index", "cwe"]].drop_duplicates()

report_df = report_df.merge(cwe_map, on="cwe_index", how="left")
# train_counts = train_df["cwe_index"].value_counts().to_dict()
val_counts = val_df["cwe_index"].value_counts().to_dict()

# report_df["train_samples"] = report_df["cwe_index"].map(train_counts)
report_df["validation_samples"] = report_df["cwe_index"].map(val_counts)
final_df = report_df[[
    "cwe",
    # "train_samples",
    "validation_samples",
    "precision",
    "recall",
    "f1-score",
    "support"
]]

final_df.rename(columns={"f1-score": "f1"}, inplace=True)
final_df.to_csv("new2.csv", index=False)
print("✅ Saved new2.csv")

/opt/homebrew/Caskroom/miniconda/base/envs/cwe/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/Caskroom/miniconda/base/envs/cwe/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/Caskroom/miniconda/base/envs/cwe/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(av

✅ Saved cwe_metrics_summary.csv


/var/folders/3z/ksnlkp8d7d12pcm61cnddf3h0000gn/T/ipykernel_5523/301845529.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df.rename(columns={"f1-score": "f1"}, inplace=True)
